# LLM chat comparison (no API)

The first two documents (one Spanish and one Catalan) were processed in the current Codex chat. The chat selected unique literal entity values for the seven labels. An exact-string alignment helper expands all occurrences and calculates their character offsets. This is a chat-plus-alignment experiment; the chat did not generate numeric offsets unaided.

The local response is `results/llm/chat_response.jsonl`. No entity text or source document is published.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'ner').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ner.evaluation import ENTITY_LABELS, evaluate_ner_strict, evaluate_ner_relaxed
from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.llm import align_chat_response
gold = load_jsonl(ROOT / 'data/plain/validation_small.jsonl')
responses = load_jsonl(ROOT / 'results/llm/chat_response.jsonl')
predictions = align_chat_response(gold, responses)
output = ROOT / 'results/llm/chat.jsonl'
save_responses_to_jsonl(predictions, output)
print('Chat documents:', len(predictions))
strict = evaluate_ner_strict(ROOT / 'data/plain/validation_small.jsonl', output)
relaxed = evaluate_ner_relaxed(ROOT / 'data/plain/validation_small.jsonl', output)

## Matched comparison

Restrict the GLiNER predictions to exactly the same two documents; the full 50-document scores are not a controlled comparison with this small chat sample.

In [ ]:
ids = {s['id'] for s in gold}
for name in ['small', 'large']:
    gliner_predictions = [p for p in load_jsonl(ROOT / f'results/plain/gliner_{name}.jsonl') if p['id'] in ids]
    matched_path = ROOT / f'results/plain/gliner_{name}_chat_subset.jsonl'
    save_responses_to_jsonl(gliner_predictions, matched_path)
    print('Model:', name)
    strict = evaluate_ner_strict(ROOT / 'data/plain/validation_small.jsonl', matched_path)
    relaxed = evaluate_ner_relaxed(ROOT / 'data/plain/validation_small.jsonl', matched_path)

Two documents cannot establish general model superiority. Exact-string alignment assists the chat with offsets; this difference must be considered when interpreting strict scores.